# Options from Zero · Episode 17: Monte Carlo pricing

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **The running trade.** An illustrative share priced at \$100 that pays no dividends, a one-year call option on it with a strike of \$105, 20% volatility and a 3.60% interest rate (continuously compounded). All made up for teaching; it is not market data.
>
> Educational material only, not financial advice. Options are risky: a seller can lose far more than the premium received.

1. Setup · 2. Last episode's question · 3. The recipe · 4. Converging · 5. The error bar · 6. Watching it converge · 7. Check against QuantLib · 8. A variance-reduction trick · 9. This episode's question · 10. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import QuantLib as ql

out = {"meta": {
    "series": "Options from Zero",
    "episode": 17,
    "quantlib_version": ql.__version__,
    "quotes_label": "illustrative (made up for teaching)",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}
S, K, T, r, vol = 100.0, 105.0, 1.0, 0.036, 0.20   # the running trade

The option formulas used throughout the series (`episodes/_shared/options.py`):

In [ ]:
# Option formulas for Options from Zero, written out by hand so every step can be read. Copied verbatim into
# each notebook by make_notebook.py, so every notebook runs on its own; the notebooks check these against
# QuantLib. Needs: math, numpy as np.
#   Running trade (Part 1): an illustrative share at $100, no dividends, 20% volatility, a one-year call struck
#   at $105, interest 3.60% a year continuously compounded. Made up for teaching.

def norm_cdf(x):
    """N(x): the chance a standard normal variable is below x."""
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def call_payoff(s, k):
    return np.maximum(np.asarray(s, dtype=float) - k, 0.0)

def put_payoff(s, k):
    return np.maximum(k - np.asarray(s, dtype=float), 0.0)

def black_scholes(s, k, t, r, vol, call=True):
    """Black-Scholes price of a European option on a share with no dividends (r continuously compounded)."""
    d1 = (math.log(s / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    d2 = d1 - vol * math.sqrt(t)
    if call:
        return s * norm_cdf(d1) - k * math.exp(-r * t) * norm_cdf(d2)
    return k * math.exp(-r * t) * norm_cdf(-d2) - s * norm_cdf(-d1)

def one_step(s, up, down, k, t, r, call=True):
    """One-step tree: replicate the option with delta shares and cash. Returns the pieces and the price."""
    pay = call_payoff if call else put_payoff
    v_up, v_down = float(pay(up, k)), float(pay(down, k))
    delta = (v_up - v_down) / (up - down)
    cash = (v_down - delta * down) * math.exp(-r * t)   # negative: borrowed today
    q = (s * math.exp(r * t) - down) / (up - down)       # risk-neutral probability of the up move
    return {"v_up": v_up, "v_down": v_down, "delta": delta, "cash": cash, "price": delta * s + cash, "q": q}

def binomial_tree(s, k, t, r, vol, steps, call=True, american=False, quantlib_p=False):
    """Cox-Ross-Rubinstein tree: u = exp(vol sqrt(dt)), d = 1/u, worked backward from expiry.
    quantlib_p=True uses QuantLib's up-probability 1/2 + 1/2 (r - vol^2/2) sqrt(dt) / vol instead of the textbook
    (e^{r dt} - d) / (u - d); both give the same price in the limit."""
    dt = t / steps
    u = math.exp(vol * math.sqrt(dt)); d = 1 / u
    p = 0.5 + 0.5 * (r - vol ** 2 / 2) * math.sqrt(dt) / vol if quantlib_p else (math.exp(r * dt) - d) / (u - d)
    pay = call_payoff if call else put_payoff
    prices = s * d ** np.arange(steps, -1, -1) * u ** np.arange(steps + 1)
    values = pay(prices, k)
    for _ in range(steps):
        prices = prices[1:] / u
        values = math.exp(-r * dt) * (p * values[1:] + (1 - p) * values[:-1])
        if american:
            values = np.maximum(values, pay(prices, k))
    return float(values[0])

In [ ]:
out["trade"] = {"spot": S, "strike": K, "years": T, "rate_pct": r * 100, "vol_pct": vol * 100}

Part 2's additions: Greeks, simulation, hedging, implied volatility, Black and Bachelier (`episodes/_shared/options_part2.py`):

In [ ]:
# Part 2 additions to options.py: the Greeks, simulation, a hedging simulator, implied volatility, and Black's and
# Bachelier's formulas on a forward (for FX and interest-rate options). Written out by hand and copied verbatim into
# each Part 2 notebook; the notebooks check them against QuantLib. Needs options.py (norm_cdf, black_scholes),
# math and numpy as np.

def norm_pdf(x):
    """The bell curve's height at x (standard normal density)."""
    return math.exp(-x * x / 2) / math.sqrt(2 * math.pi)

def norm_cdf_v(x):
    """norm_cdf for a whole numpy array at once."""
    return 0.5 * (1 + np.vectorize(math.erf)(np.asarray(x, dtype=float) / math.sqrt(2)))

def bs_greeks(s, k, t, r, vol):
    """Price and Greeks of a European call on a share with no dividends.
    delta per $1 of share price, gamma per $1 squared, vega per 1.00 of volatility (100 vol points),
    theta per year (negative: value lost as time passes), rho per 1.00 of interest rate."""
    d1 = (math.log(s / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    d2 = d1 - vol * math.sqrt(t)
    disc = math.exp(-r * t)
    return {"price": s * norm_cdf(d1) - k * disc * norm_cdf(d2),
            "delta": norm_cdf(d1),
            "gamma": norm_pdf(d1) / (s * vol * math.sqrt(t)),
            "vega": s * norm_pdf(d1) * math.sqrt(t),
            "theta": -s * norm_pdf(d1) * vol / (2 * math.sqrt(t)) - r * k * disc * norm_cdf(d2),
            "rho": k * t * disc * norm_cdf(d2)}

def call_delta(s, k, t, r, vol):
    """Black-Scholes delta N(d1) for an array of share prices."""
    d1 = (np.log(np.asarray(s, dtype=float) / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    return norm_cdf_v(d1)

def simulate_paths(s, t, drift, vol, steps, n_paths, rng):
    """Share price paths: shape (n_paths, steps + 1), starting at s, lognormal steps with the given drift."""
    dt = t / steps
    z = rng.standard_normal((n_paths, steps))
    log_steps = (drift - vol ** 2 / 2) * dt + vol * math.sqrt(dt) * z
    return s * np.exp(np.concatenate([np.zeros((n_paths, 1)), np.cumsum(log_steps, axis=1)], axis=1))

def monte_carlo_call(s, k, t, r, vol, n, rng):
    """Price = the average discounted payoff over n risk-neutral outcomes, with its standard error."""
    z = rng.standard_normal(n)
    s_t = s * np.exp((r - vol ** 2 / 2) * t + vol * math.sqrt(t) * z)
    pv = math.exp(-r * t) * np.maximum(s_t - k, 0.0)
    return float(pv.mean()), float(pv.std(ddof=1) / math.sqrt(n))

def hedge_short_call(s, k, t, r, vol_implied, vol_realised, steps, n_paths, rng, hedge=True):
    """Sell the call at vol_implied and, if hedge, rebalance to delta shares (computed at vol_implied) `steps` times.
    The share moves with vol_realised and grows at r. Cash earns r. Returns each path's P&L at expiry in today's dollars."""
    dt = t / steps
    price = s * np.ones(n_paths)
    held = call_delta(price, k, t, r, vol_implied) if hedge else np.zeros(n_paths)
    cash = black_scholes(s, k, t, r, vol_implied) - held * price      # premium received, shares bought
    for i in range(1, steps + 1):
        price = price * np.exp((r - vol_realised ** 2 / 2) * dt + vol_realised * math.sqrt(dt) * rng.standard_normal(n_paths))
        cash = cash * math.exp(r * dt)
        if hedge and i < steps:
            new = call_delta(price, k, t - i * dt, r, vol_implied)
            cash = cash - (new - held) * price                        # buy (or sell) the difference
            held = new
    pnl = cash + held * price - np.maximum(price - k, 0.0)           # sell the shares, pay the option's payoff
    return pnl * math.exp(-r * t)

def implied_vol(price, s, k, t, r, lo=0.0001, hi=3.0, tol=1e-12):
    """Run Black-Scholes backwards by bisection: halve [lo, hi] until the price fits. Returns (vol, halvings)."""
    n = 0
    while hi - lo > tol:
        mid = (lo + hi) / 2
        if black_scholes(s, k, t, r, mid) < price:
            lo = mid
        else:
            hi = mid
        n += 1
    return (lo + hi) / 2, n

def black76(forward, strike, t, vol, df, call=True):
    """Black's formula on a forward: df x [F N(d1) - K N(d2)] for a call. FX options (Garman-Kohlhagen) and caplets."""
    sd = vol * math.sqrt(t)
    d1 = (math.log(forward / strike) + sd ** 2 / 2) / sd
    d2 = d1 - sd
    if call:
        return df * (forward * norm_cdf(d1) - strike * norm_cdf(d2))
    return df * (strike * norm_cdf(-d2) - forward * norm_cdf(-d1))

def bachelier(forward, strike, t, normal_vol, df, call=True):
    """Bachelier's formula: the forward moves by normal_vol x sqrt(t) in absolute terms (it can go below zero)."""
    sd = normal_vol * math.sqrt(t)
    d = (forward - strike) / sd
    if call:
        return df * ((forward - strike) * norm_cdf(d) + sd * norm_pdf(d))
    return df * ((strike - forward) * norm_cdf(-d) + sd * norm_pdf(d))

## 2. Last episode's question

Share flat, volatility up one point, one day passes: vega per point plus a day's theta, about $+0.399 - 0.015 = +0.38$.

In [ ]:
g = bs_greeks(S, K, T, r, vol)
out["answer"] = {"vega_pt": g["vega"] / 100, "theta_day_abs": abs(g["theta"]) / 365, "approx": g["vega"] / 100 + g["theta"] / 365}
out["answer"]

## 3. The recipe

Route three to the price (previewed here, finished in Part 3): the price is the risk-neutral average of the discounted payoff (Episode 6). So simulate it:

1. Draw the share price at expiry many times, growing at the risk-free rate $r$ with volatility $\sigma$: $S_T = S\,e^{(r - \sigma^2/2)T + \sigma\sqrt{T}Z}$.
2. Pay $\max(S_T - K, 0)$ on each.
3. Discount and average.

Some paths, to see it (monthly steps):

In [ ]:
paths = simulate_paths(S, T, r, vol, 52, 60, np.random.default_rng(171))
t = np.linspace(0, T, 53)
ends = simulate_paths(S, T, r, vol, 1, 50_000, np.random.default_rng(172))[:, -1]
bins = np.arange(50.0, 190.01, 5.0)
counts, _ = np.histogram(np.clip(ends, 50, 189.99), bins=bins)
out["fan"] = {"t": t.round(4).tolist(), "paths": paths.round(2).tolist(),
              "hist": {"y": ((bins[:-1] + bins[1:]) / 2).tolist(), "share": (counts / len(ends)).round(5).tolist()},
              "itm_share": float((ends > K).mean()),
              "ticks": [{"x": x, "label": lab} for x, lab in ((0, "0"), (0.25, "3m"), (0.5, "6m"), (0.75, "9m"), (1, "1y"))]}
plt.plot(t, paths.T, lw=0.6); plt.axhline(K, color="k", ls="--"); plt.show()

## 4. Converging

Each estimate comes with a **standard error**: the standard deviation of the discounted payoffs divided by $\sqrt{N}$. It shrinks like $1/\sqrt N$: 100 times more paths buys 10 times more accuracy.

In [ ]:
rng = np.random.default_rng(17)
formula = black_scholes(S, K, T, r, vol)
rows = []
for n in (1_000, 10_000, 100_000, 1_000_000):
    price, se = monte_carlo_call(S, K, T, r, vol, n, rng)
    rows.append({"paths": n, "price": price, "se": se, "z": (price - formula) / se, "se_root_n": se * math.sqrt(n),
                 "error": price - formula})
assert all(abs(row["z"]) < 3 for row in rows)
out["mc"] = {"rows": rows, "formula": formula, "worst_z": max(abs(row["z"]) for row in rows),
             "se_ratio_10x": rows[1]["se"] / rows[0]["se"],
             "se_first": rows[0]["se"], "se_last": rows[-1]["se"], "price_first": rows[0]["price"], "price_last": rows[-1]["price"]}
pd.DataFrame(rows)

## 5. The error bar

Roughly 95% of estimates land within two standard errors of the true price. The column $SE\times\sqrt N$ is constant: it is the spread of a single discounted payoff, about \$13, so the standard error for $N$ paths is about $13/\sqrt N$. To get $\pm\$0.01$ you would need well over a million paths.

In [ ]:
spread = float(np.mean([row["se_root_n"] for row in rows]))
out["error_bar"] = {"payoff_sd": spread, "paths_for_1c": (spread / 0.01) ** 2, "paths_for_1c_m": round((spread / 0.01) ** 2 / 1e6, 1),
                    "within_2se": sum(abs(row["z"]) < 2 for row in rows)}
out["error_bar"]

## 6. Watching it converge

One long run: the running average as paths are added, against the formula, with a band of ±2 standard errors.

In [ ]:
z = np.random.default_rng(1717).standard_normal(1_000_000)
pv = math.exp(-r * T) * np.maximum(S * np.exp((r - vol ** 2 / 2) * T + vol * math.sqrt(T) * z) - K, 0.0)
cum = np.cumsum(pv) / np.arange(1, len(pv) + 1)
ns = np.unique(np.logspace(2, 6, 160).astype(int))
x = np.log10(ns)
band = 2 * pv.std() / np.sqrt(ns)
out["run"] = {"x": x.round(4).tolist(), "estimate": cum[ns - 1].tolist(), "formula": [formula] * len(ns),
              "upper": (formula + band).tolist(), "lower": (formula - band).tolist(), "final": float(cum[-1]),
              "ticks": [{"x": float(k), "label": lab} for k, lab in ((2, "100"), (3, "1,000"), (4, "10,000"), (5, "100,000"), (6, "1,000,000"))]}
plt.semilogx(ns, cum[ns - 1]); plt.axhline(formula, color="k"); plt.semilogx(ns, formula + band, "k:", ns, formula - band, "k:"); plt.show()

## 7. Check against QuantLib

QuantLib's `MCEuropeanEngine` does the same (pseudo-random numbers, one time step, fixed seed).

In [ ]:
today = ql.Date(29, 9, 2026)
ql.Settings.instance().evaluationDate = today
dc = ql.Actual365Fixed()
flat = lambda x: ql.YieldTermStructureHandle(ql.FlatForward(today, x, dc, ql.Continuous))
process = ql.BlackScholesMertonProcess(
    ql.QuoteHandle(ql.SimpleQuote(S)), flat(0.0), flat(r),
    ql.BlackVolTermStructureHandle(ql.BlackConstantVol(today, ql.NullCalendar(), vol, dc)))
expiry = today + 365   # exactly one year on ACT/365F
opt = ql.VanillaOption(ql.PlainVanillaPayoff(ql.Option.Call, K), ql.EuropeanExercise(expiry))
opt.setPricingEngine(ql.MCEuropeanEngine(process, "pseudorandom", timeSteps=1, requiredSamples=100_000, seed=42))
out["quantlib"] = {"price": opt.NPV(), "se": opt.errorEstimate(), "z": (opt.NPV() - formula) / opt.errorEstimate()}
assert abs(out["quantlib"]["z"]) < 3
out["quantlib"]

## 8. A variance-reduction trick (notebook extra)

**Antithetic variates**: use each draw $Z$ twice, as $Z$ and $-Z$, and average the pair. The two payoffs move in opposite directions, so the average wobbles less: the same accuracy for fewer draws.

In [ ]:
z = np.random.default_rng(1718).standard_normal(50_000)
pay = lambda zz: math.exp(-r * T) * np.maximum(S * np.exp((r - vol ** 2 / 2) * T + vol * math.sqrt(T) * zz) - K, 0.0)
pairs = 0.5 * (pay(z) + pay(-z))
out["antithetic"] = {"pairs": len(pairs), "evaluations": 2 * len(pairs), "price": float(pairs.mean()), "se": float(pairs.std(ddof=1) / math.sqrt(len(pairs))),
                     "plain_se_same_draws": float(pay(np.concatenate([z, np.random.default_rng(1719).standard_normal(50_000)])).std(ddof=1) / math.sqrt(100_000))}
out["antithetic"]

## 9. This episode's question

An error of ±0.40 with 1,000 paths: for ±0.04, ten times smaller, you need $10^2 = 100$ times as many paths, 100,000.

In [ ]:
out["question"] = {"paths": 1_000, "se": 0.40, "target": 0.04, "needed": 1_000 * (0.40 / 0.04) ** 2}
out["question"]

## 10. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")